<a href="https://colab.research.google.com/github/sibandze/Bird-Intelligence-System/blob/dev-unsupervised/notebooks/exploration_and_data_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Notebook: Data Loading + Spectogram Pipeline + Exploration

---



**Mount Google Drive & Define Directories**

In [ ]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Master Google Drive folder for dataset backups
DRIVE_BACKUP_DIR = '/content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs'
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
print(f"Drive backup directory ready: {DRIVE_BACKUP_DIR}")

Mounted at /content/drive
Drive backup directory ready: /content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs


**Clone Repository & Install Dependencies**

In [ ]:
import os
import sys

REPO_URL = "https://github.com/sibandze/Bird-Intelligence-System.git"
BRANCH = "dev-unsupervised"
REPO_DIR = "/content/Bird-Intelligence-System"

if not os.path.exists(REPO_DIR):
    print("Cloning repo...")
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print("Repo exists. Pulling latest...")
    %cd {REPO_DIR}
    !git fetch origin {BRANCH}
    !git reset --hard origin/{BRANCH}  # this overwrites local changes

%cd {REPO_DIR}

if os.path.exists("requirements.txt"):
    !pip install -r requirements.txt

Cloning repo...
Cloning into '/content/Bird-Intelligence-System'...
remote: Enumerating objects: 1853, done.
remote: Counting objects: 100% (248/248), done.
remote: Compressing objects: 100% (184/184), done.
remote: Total 1853 (delta 146), reused 94 (delta 57), pack-reused 1605 (from 3)
Receiving objects: 100% (1853/1853), 346.58 MiB | 22.73 MiB/s, done.
Resolving deltas: 100% (903/903), done.
/content/Bird-Intelligence-System


Load Config & Restore Existing Files from Google Drive

In [ ]:
import re
import shutil
import yaml

CONFIG_PATH = os.path.join(REPO_DIR, "configs", "config.yaml")

with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

# Extract relative paths from config
raw_audio_dir = config['data']['raw_audio_dir']             # data/raw_audio
processed_npy_dir = config['data']['processed_npy_dir']     # data/processed_spectrograms
metadata_dir = config['data']['metadata_dir']               # data/metadata

# Ensure target local directories exist inside the cloned repo
for rel_dir in [raw_audio_dir, processed_npy_dir, metadata_dir]:
    os.makedirs(os.path.join(REPO_DIR, rel_dir), exist_ok=True)

# Audio & Spectrogram Config Parameters to match against file naming schemes
audio_params = config.get('audio', {})
sr = audio_params.get('sr', 32000)
n_fft = audio_params.get('n_fft', 2048)
hop_length = audio_params.get('hop_length', 512)
n_mels = audio_params.get('n_mels', 128)

# Expected parameter string in processed file names (e.g., _sr32000_nfft2048_hop512_nmel128_)
spectrogram_pattern = f"sr{sr}_nfft{n_fft}_hop{hop_length}_nmel{n_mels}"

print(f"Scanning Drive backup using pattern key: '{spectrogram_pattern}'...\n")

def restore_files_from_drive(target_rel_dir, file_filter_fn):
    drive_dir = os.path.join(DRIVE_BACKUP_DIR, target_rel_dir)
    local_dir = os.path.join(REPO_DIR, target_rel_dir)

    restored_count = 0
    if os.path.exists(drive_dir):
        for filename in os.listdir(drive_dir):
            drive_file_path = os.path.join(drive_dir, filename)
            local_file_path = os.path.join(local_dir, filename)

            if os.path.isfile(drive_file_path) and file_filter_fn(filename):
                if not os.path.exists(local_file_path):
                    shutil.copy2(drive_file_path, local_file_path)
                    restored_count += 1

        print(f"[{target_rel_dir}] Restored {restored_count} matching file(s) from Drive.")
    else:
        print(f"[{target_rel_dir}] No Drive directory found at {drive_dir}")

# 1. Restore Raw Audio (.ogg / .wav files)
restore_files_from_drive(
    raw_audio_dir,
    lambda f: f.endswith(('.ogg', '.wav', '.mp3'))
)

# 2. Restore Spectrograms matching current audio parameters
restore_files_from_drive(
    processed_npy_dir,
    lambda f: f.endswith('.npy') and spectrogram_pattern in f
)

# 3. Restore Metadata files (.csv)
restore_files_from_drive(
    metadata_dir,
    lambda f: f.endswith('.csv') and spectrogram_pattern in f
)

# Optional: Override/update runtime config parameters
config['training']['batch_size'] = 32
config['training']['epochs'] = 50
config['logging']['use_wandb'] = False

RUN_CONFIG_PATH = os.path.join(REPO_DIR, "configs", "config_colab_run.yaml")
with open(RUN_CONFIG_PATH, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print("\nConfiguration resolved and Drive files restored to local workspace.")

Scanning Drive backup using pattern key: 'sr32000_nfft2048_hop512_nmel128'...

[data/raw_audio] No Drive directory found at /content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs/data/raw_audio
[data/processed_spectrograms] No Drive directory found at /content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs/data/processed_spectrograms
[data/metadata] No Drive directory found at /content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs/data/metadata

Configuration resolved and Drive files restored to local workspace.


In [ ]:
# Execute the pipeline runner module with the updated config
!python -m pipeline.pipeline_runner --config configs/config_colab_run.yaml --full-dataset

✓ segment_seconds=3s -> 187 frames -> segment_size=200 (3.20s) [patch=25]
Starting Data Pipeline

Loading metadata...
🌐 Mode: Full Dataset (No balancing/class filtering)

Metadata summary
----------------
Total rows in source : 2,161
Selected classes     : 219
Target total samples : 2,161

Processing audio:   0% 0/2161 [00:00<?, ?file/s]Downloaded: /content/Bird-Intelligence-System/data/raw_audio/XC516153.ogg
Processing audio:   0% 1/2161 [00:04<2:46:55,  4.64s/file, fail=0, proc=1, skip=0]Downloaded: /content/Bird-Intelligence-System/data/raw_audio/XC208209.ogg
Processing audio:   0% 2/2161 [00:06<1:38:12,  2.73s/file, fail=0, proc=2, skip=0]Downloaded: /content/Bird-Intelligence-System/data/raw_audio/XC208128.ogg
Processing audio:   0% 3/2161 [00:06<1:07:00,  1.86s/file, fail=0, proc=3, skip=0]Downloaded: /content/Bird-Intelligence-System/data/raw_audio/XC46725.ogg
Processing audio:   0% 4/2161 [00:07<53:32,  1.49s/file, fail=0, proc=4, skip=0]  Error downloading https://xeno-canto.o

In [ ]:
import pandas as pd
import tarfile

# The pipeline runner generated a metadata file with a specific name.
config['data']['data_csv'] = f"metadata_full_{spectrogram_pattern}.csv"

# Construct the full path to the metadata file using the updated config
metadata_file_path = os.path.join(REPO_DIR, metadata_dir, config['data']['data_csv'])

# Load the metadata file
metadata_df = pd.read_csv(metadata_file_path)
display(metadata_df.head())

,common_name,scientific_name,Download_link,xc_id,scientific_name_id,spectrogram_filename,local_spectrogram_path,total_frames,rc_id,num_windows
0,Common Ostrich,Struthio camelus australis,https://xeno-canto.org/516153/download,XC516153,0,XC516153_sr32000_nfft2048_hop512_nmel128_seg20...,/content/Bird-Intelligence-System/data/process...,3349,XC516153,14
1,Common Ostrich,Struthio camelus,https://xeno-canto.org/208209/download,XC208209,1,XC208209_sr32000_nfft2048_hop512_nmel128_seg20...,/content/Bird-Intelligence-System/data/process...,1672,XC208209,7
2,Common Ostrich,Struthio camelus,https://xeno-canto.org/208128/download,XC208128,1,XC208128_sr32000_nfft2048_hop512_nmel128_seg20...,/content/Bird-Intelligence-System/data/process...,292,XC208128,2
3,Common Ostrich,Struthio camelus,https://xeno-canto.org/46725/download,XC46725,1,XC46725_sr32000_nfft2048_hop512_nmel128_seg200...,/content/Bird-Intelligence-System/data/process...,737,XC46725,4
4,Common Ostrich,Struthio camelus,https://xeno-canto.org/673753/download,XC673753,1,XC673753_sr32000_nfft2048_hop512_nmel128_seg20...,/content/Bird-Intelligence-System/data/process...,729,XC673753,4


Next, I will create a dictionary mapping each species to a list of its corresponding raw audio file paths.

In [ ]:
# Define a dictionary to store audio file paths grouped by species
species_audio_files = {}

# Construct the full path to the raw audio directory
full_raw_audio_dir = os.path.join(REPO_DIR, raw_audio_dir)

for index, row in metadata_df.iterrows():
    species = row['scientific_name']
    filename = f'{row['rc_id']}.ogg' #row['filename'] # Assuming 'filename' is the column with audio filenames
    audio_file_path = os.path.join(full_raw_audio_dir, filename)

    if os.path.exists(audio_file_path):
        if species not in species_audio_files:
            species_audio_files[species] = []
        species_audio_files[species].append(audio_file_path)
    else:
        print(f"Warning: Audio file not found for {species}: {audio_file_path}")

print(f"Found {len(species_audio_files)} unique species with audio files.")
# Display a sample of the dictionary
for species, files in list(species_audio_files.items())[:3]:
    print(f"\nSpecies: {species}")
    for f in files[:5]: # Display up to 5 files per species for brevity
        print(f"  - {f}")

Found 216 unique species with audio files.

Species: Struthio camelus australis
  - /content/Bird-Intelligence-System/data/raw_audio/XC516153.ogg

Species: Struthio camelus
  - /content/Bird-Intelligence-System/data/raw_audio/XC208209.ogg
  - /content/Bird-Intelligence-System/data/raw_audio/XC208128.ogg
  - /content/Bird-Intelligence-System/data/raw_audio/XC46725.ogg
  - /content/Bird-Intelligence-System/data/raw_audio/XC673753.ogg
  - /content/Bird-Intelligence-System/data/raw_audio/XC563003.ogg

Species: Struthio molybdophanes
  - /content/Bird-Intelligence-System/data/raw_audio/XC292043.ogg


Finally, I will create a tar.gz archive for the audio files of each species. These archives will be saved in the Drive backup directory under a new `archived_audio` folder.

In [ ]:
# Create a directory for the archived audio files in the Drive backup
archive_output_dir = os.path.join(DRIVE_BACKUP_DIR, 'archived_audio')
os.makedirs(archive_output_dir, exist_ok=True)

for species, files in species_audio_files.items():
    # Sanitize species name for filename (replace problematic characters)
    sanitized_species = species.replace(' ', '_').replace('/', '_').replace('(', '').replace(')', '').lower()
    archive_filename = f"{sanitized_species}_audio.tar.gz"
    archive_path = os.path.join(archive_output_dir, archive_filename)

    if not files:
        print(f"Skipping {species}: no audio files found.")
        continue

    with tarfile.open(archive_path, "w:gz") as tar:
        for f_path in files:
            # Add the file to the archive, preserving its base name
            tar.add(f_path, arcname=os.path.basename(f_path))
    print(f"Created archive for {species} at: {archive_path}")

print("\nAll species audio files have been archived.")

Created archive for Struthio camelus australis at: /content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs/archived_audio/struthio_camelus_australis_audio.tar.gz
Created archive for Struthio camelus at: /content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs/archived_audio/struthio_camelus_audio.tar.gz
Created archive for Struthio molybdophanes at: /content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs/archived_audio/struthio_molybdophanes_audio.tar.gz
Created archive for Rhea americana at: /content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs/archived_audio/rhea_americana_audio.tar.gz
Created archive for Rhea americana araneipes at: /content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs/archived_audio/rhea_americana_araneipes_audio.tar.gz
Created archive for Rhea americana albescens at: /content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs/archived_audio/rhea_americana_albescens_audio.tar.gz
Created archive for Rhea americana

In [ ]:
# Create a directory for the archived spectrogram files in the Drive backup
archive_spectrogram_output_dir = os.path.join(DRIVE_BACKUP_DIR, 'archived_spectrograms')
os.makedirs(archive_spectrogram_output_dir, exist_ok=True)

for species, _ in species_audio_files.items(): # Iterate through species using the keys from the audio files dictionary
    # Filter metadata_df for the current species to get all relevant spectrogram filenames
    species_df = metadata_df[metadata_df['scientific_name'] == species]

    spectrogram_files_to_archive = []
    for _, row in species_df.iterrows():
        spectrogram_filename = row['spectrogram_filename']
        spectrogram_file_path = os.path.join(REPO_DIR, processed_npy_dir, spectrogram_filename)
        if os.path.exists(spectrogram_file_path):
            spectrogram_files_to_archive.append(spectrogram_file_path)
        else:
            print(f"Warning: Spectrogram file not found for {species}: {spectrogram_file_path}")

    if not spectrogram_files_to_archive:
        print(f"Skipping {species}: no spectrogram files found.")
        continue

    # Sanitize species name for filename (replace problematic characters)
    sanitized_species = species.replace(' ', '_').replace('/', '_').replace('(', '').replace(')', '').lower()
    archive_filename = f"{sanitized_species}_spectrograms_{spectrogram_pattern}.tar.gz"
    archive_path = os.path.join(archive_spectrogram_output_dir, archive_filename)

    with tarfile.open(archive_path, "w:gz") as tar:
        for f_path in spectrogram_files_to_archive:
            # Add the file to the archive, preserving its base name
            tar.add(f_path, arcname=os.path.basename(f_path))
    print(f"Created archive for {species} at: {archive_path}")

print("\nAll species spectrogram files have been archived.")

Created archive for Struthio camelus australis at: /content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs/archived_spectrograms/struthio_camelus_australis_spectrograms_sr32000_nfft2048_hop512_nmel128.tar.gz
Created archive for Struthio camelus at: /content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs/archived_spectrograms/struthio_camelus_spectrograms_sr32000_nfft2048_hop512_nmel128.tar.gz
Created archive for Struthio molybdophanes at: /content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs/archived_spectrograms/struthio_molybdophanes_spectrograms_sr32000_nfft2048_hop512_nmel128.tar.gz
Created archive for Rhea americana at: /content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs/archived_spectrograms/rhea_americana_spectrograms_sr32000_nfft2048_hop512_nmel128.tar.gz
Created archive for Rhea americana araneipes at: /content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs/archived_spectrograms/rhea_americana_araneipes_spectrograms_sr32000_nf

In [ ]:
metadata_source_path = metadata_file_path
drive_metadata_backup_dir = os.path.join(DRIVE_BACKUP_DIR, metadata_dir)
os.makedirs(drive_metadata_backup_dir, exist_ok=True)
metadata_destination_path = os.path.join(drive_metadata_backup_dir, os.path.basename(metadata_source_path))
shutil.copy2(metadata_source_path, metadata_destination_path)

print(f"Backed up metadata file to: {metadata_destination_path}")